# 05 · Actuarial Analyst Agent

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S2.P3  ·  **Slides:** S2.P3.3–8
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Kasyap
**Date:** 24 July 2026 · Four Points by Sheraton, Whitefield, Bangalore
**Model:** `gemini-3.1-flash-lite` (pinned)  ·  **License:** CC BY-NC 4.0

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks_v3/05_actuarial_analyst_agent.ipynb)

## What this notebook does
The capstone: this morning's models become an agent's tools; RAG grounds it in ABC methodology; a reviewer agent (Arjun's checklist) recomputes and challenges.

*All data is hypothetical — ABC Insurer is a fictional entity for teaching only.
The story: Priya Nair (pricing, ABC General) must explain the price of policy
**ABC-MOT-047231** — a 7-year-old SUV, Tier-2, 35% NCB — so her chief actuary
**Arjun Mehta** can sign it.*

In [ ]:
%pip install -q agno google-genai scikit-learn

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    assert "GOOGLE_API_KEY" in os.environ
MODEL = "gemini-3.1-flash-lite"

## 1. This morning's pipeline as tools
(Stubs return the notebook-02 results; wire your real functions here for the case study.)

In [ ]:
def load_data() -> str:
    """Load & schema-validate ABC Motor 2024."""
    return "ABC Motor 2024: 5000 rows, schema OK."

def fit_glm() -> str:
    """Fit the Poisson GLM; return bias."""
    return "GLM fitted. Bias 0.0%."

def fit_xgboost() -> str:
    """Fit the XGBoost challenger; return top-decile lift."""
    return "XGBoost fitted. Top-decile lift 3.8x on Q4-2024."

def lift_table() -> str:
    """Out-of-time lift comparison, test window Q4-2024."""
    return "Lift (Q4-2024): GLM 2.9x, XGBoost 3.8x."

def shap_explain() -> str:
    """SHAP attribution for ABC-MOT-047231."""
    return "047231: base 8.2% -> +age -> -NCB -> 8.9%."

def summarise_results() -> str:
    """Draft a board memo from the structured results."""
    return "Drafted."

## 2. The capstone agent — one instruction, five tools

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

ANALYST_SYSTEM = ("You are an actuarial analyst. Use ONLY tool outputs for numbers. "
    "Order: data, then models, then lift, then SHAP, then summary. "
    "Always name the test window. Always report fairness.")

analyst = Agent(model=Gemini(id=MODEL),
    tools=[load_data, fit_glm, fit_xgboost, lift_table, shap_explain, summarise_results],
    system_message=ANALYST_SYSTEM, show_tool_calls=True)

analyst.print_response("Fit, compare and explain a frequency model for ABC Motor 2024. Draft the memo.")

## 3. RAG — the retriever is just another tool

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DOCS = {
 "LAPSE-2025-03": "ABC Life term lapse uses a select-and-ultimate basis; year-2 lapse 12%.",
 "PRICE-2025-01": "ABC Motor pricing uses earned exposure and out-of-time Q4 testing.",
 "FAIR-2025-02":  "ABC audits calibration-within-subgroup on gender x age band before ship.",
}
_ids = list(DOCS); _vec = TfidfVectorizer().fit(DOCS.values())
_mat = _vec.transform(DOCS.values())

def search_methodology(query: str) -> list:
    """Retrieve the ABC methodology snippets most relevant to a query. Cite the id."""
    sims = cosine_similarity(_vec.transform([query]), _mat)[0]
    return [(_ids[i], list(DOCS.values())[i]) for i in sims.argsort()[::-1][:2]]

# Rebuild the analyst WITH the retriever registered as a tool (appending after
# construction isn't guaranteed to re-register the tool across Agno versions).
analyst = Agent(model=Gemini(id=MODEL),
    tools=[load_data, fit_glm, fit_xgboost, lift_table, shap_explain,
           summarise_results, search_methodology],
    system_message=ANALYST_SYSTEM, show_tool_calls=True)
analyst.print_response("What lapse basis do we use for term business? Cite the document.")

## 4. Multi-agent — the reviewer is Arjun's checklist

In [ ]:
def recompute_lift() -> str:
    """Independently recompute the out-of-time lift table."""
    return "Recomputed lift (Q4-2024): XGBoost 3.8x — confirmed."

def recompute_fairness() -> str:
    """Independently rerun the calibration-within-subgroup check."""
    return "Fairness: within 1pt in all cells — pass."

CHECKLIST = ("You are a sceptical reviewing actuary. Apply the 10-question checklist. "
    "Recompute lift and fairness with your tools; never trust the draft's numbers. "
    "Flag any figure quoted without its test window.")

reviewer = Agent(model=Gemini(id=MODEL),
    tools=[recompute_lift, recompute_fairness], system_message=CHECKLIST, show_tool_calls=True)

reviewer.print_response("Review this draft: 'The XGBoost model achieves 3.8x lift and is fair.'")

## TODO
Add a `search_methodology` rule so the analyst **cites** a document whenever it states a methodology choice. Then have the reviewer fail any uncited methodology claim.

**MCP note:** wrap `fit_glm` in an MCP server and any teammate's agent (or Claude Desktop) can call it.

## Wrap-up
*Demonstrated: one instruction runs the whole pipeline; RAG grounds answers in your docs; a reviewer agent recomputes rather than opines — the actuarial control cycle in silicon.*